SHASHANK BHARAGANNARA GIRISWAMY

20022954

Programming assignment 2

### 1. Convolutional Layer Operations
The fundamental operation in our CNN is the convolution, defined mathematically as:

(f * g)[n] = \sum_{m=-\infty}^{\infty} f[m]g[n-m]

For 2D images, this becomes:

(I * K)[i,j] = \sum_{m}\sum_{n} I[m,n]K[i-m,j-n]

where:
- I is the input image
- K is the kernel/filter
- [i,j] are the pixel coordinates

### 2. Activation Functions
We use ReLU (Rectified Linear Unit) activation:

ReLU(x) = max(0,x)

Chosen because:
- Prevents vanishing gradient problem
- Computationally efficient
- Promotes sparsity in the network

### 3. Pooling Operations
MaxPooling with 2×2 windows:

MaxPool(x)_{i,j} = max_{m,n \in R_{i,j}} x_{m,n}

where R_{i,j} is the 2×2 region with anchor point (i,j)

### 4. Loss Function
Mean Squared Error (MSE):

MSE = \frac{1}{n}\sum_{i=1}^n(y_i - \hat{y}_i)^2

where:
- y_i is the true count
- hat{y}_i is the predicted count


In [53]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import zipfile
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
import pandas as pd
from pathlib import Path

In [54]:
# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

In [55]:
def extract_dataset():
   # Extracting the dataset from the specified ZIP file path
    
    zip_path = r"C:\Users\Shashank B G\OneDrive\Desktop\CS_583_DL\TomatoPlantfactoryDataset.zip"
    extract_path = os.path.dirname(zip_path)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    
    return os.path.join(extract_path, "TomatoPlantfactoryDataset")

In [56]:
def load_and_preprocess_image(image_path, target_size=(224, 224)):

    #Load and preprocess a single image

    # Reading image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resizeing the images
    img = cv2.resize(img, target_size)
    
    # Normalizing to [0, 1]
    img = img.astype(np.float32) / 255.0
    
    return img


In [57]:
def data_augmentation(img):
    #Apply data augmentation to a single image
    
    # Randomnly rotating the images
    if np.random.random() > 0.5:
        angle = np.random.uniform(-20, 20)
        height, width = img.shape[:2]
        matrix = cv2.getRotationMatrix2D((width/2, height/2), angle, 1.0)
        img = cv2.warpAffine(img, matrix, (width, height))
    
    # Randomnly horizontal flip
    if np.random.random() > 0.5:
        img = cv2.flip(img, 1)
    
    # Randomnly brightness adjustment
    if np.random.random() > 0.5:
        img = tf.image.random_brightness(img, 0.2)
    
    return img

# CNN Architecture Design
Our CNN architecture follows a proven design pattern for object detection:

1. Initial Convolution Block
   - 32 3×3 filters capture basic features
   - MaxPooling reduces spatial dimensions
   - BatchNormalization stabilizes training

2. Deeper Convolution Blocks
   - Increased filter count (64, 128) captures complex features
   - Maintains spatial hierarchy through pooling

3. Dense Layers
   - Flatten layer converts spatial features to 1D
   - Dense layers learn high-level counting relationships
   - Dropout prevents overfitting

In [58]:
def load_dataset(dataset_path):
    #Load all images and their corresponding tomato counts
    
    images = []
    counts = []
    
    # Assuming the annotations are in a CSV file named 'annotations.csv'
    # with columns 'image_name' and 'tomato_count'
    annotations_df = pd.read_csv(os.path.join(dataset_path, 'annotations.csv'))
    
    for idx, row in annotations_df.iterrows():
        img_path = os.path.join(dataset_path, 'images', row['image_name'])
        if os.path.exists(img_path):
            img = load_and_preprocess_image(img_path)
            images.append(img)
            counts.append(row['tomato_count'])
    
    return np.array(images), np.array(counts)


In [59]:
def create_cnn_model():
    
    #Create the CNN model architecture as specified in the assignment
   
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Third Convolutional Block (additional)
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Flatten and Dense Layers
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(1)  # Output layer for count prediction
    ])
    
    return model


In [60]:
def train_model(model, X_train, y_train, X_val, y_val):
    
    #Train the model with the specified parameters
   
    # Compile the model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    # Create early stopping callback
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_mae',
        patience=5,
        restore_best_weights=True
    )
    
    # Train the model
    history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping]
    )
    
    return history

In [61]:
def evaluate_model(model, X_test, y_test):
    
    #Evaluate the model and calculate required metrics
   
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = np.mean(np.abs(y_pred - y_test))
    rmse = np.sqrt(np.mean((y_pred - y_test) ** 2))
    
    return mae, rmse, y_pred


In [62]:
def plot_results(history, y_test, y_pred):
    
    #Create and save visualization plots
  
    # Plot training history
    plt.figure(figsize=(12, 4))
    
    # Loss curves
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss Over Time')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Predicted vs Actual
    plt.subplot(1, 2, 2)
    plt.scatter(y_test, y_pred)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.title('Predicted vs Actual Tomato Counts')
    plt.xlabel('Actual Count')
    plt.ylabel('Predicted Count')
    
    plt.tight_layout()
    plt.savefig('training_results.png')
    plt.close()


 Model Results Discussion

Performance Analysis
1. Metrics Interpretation
   - MAE: Measures average counting error
   - RMSE: Penalizes larger counting errors more heavily
   - Loss curves indicate model convergence

2. Model Behavior
   - Early epochs show rapid improvement
   - Validation metrics stabilize around epoch X
   - Final performance metrics:
     * MAE: X tomatoes
     * RMSE: Y tomatoes

 Challenges Encountered
1. Data Challenges
   - Varying lighting conditions affecting feature detection
   - Occlusion between tomatoes
   - Different growth stages affecting appearance

2. Model Challenges
   - Balancing model capacity with overfitting
   - Handling extreme cases (very few or many tomatoes)
   - Computational resources for training

Potential Improvements
1. Architecture Enhancements
   - Consider ResNet-style skip connections
   - Implement attention mechanisms
   - Try different backbone architectures

2. Data Improvements
   - Additional augmentation techniques
   - Collect more diverse training data
   - Implement weighted sampling for rare cases

3. Training Optimizations
   - Learning rate scheduling
   - Cross-validation for hyperparameter tuning
   - Ensemble methods


In [ ]:
def main():
    
    #Main execution function
    
    # Extract and load dataset
    dataset_path = extract_dataset()
    X, y = load_dataset(dataset_path)
    
    # Split dataset
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
    
    # Create and train model
    model = create_cnn_model()
    history = train_model(model, X_train, y_train, X_val, y_val)
    
    # Evaluate model
    mae, rmse, y_pred = evaluate_model(model, X_test, y_test)
    
    # Plot and save results
    plot_results(history, y_test, y_pred)
    
    # Save results to file
    with open('model_results.txt', 'w') as f:
        f.write(f"Model Evaluation Results\n")
        f.write(f"Mean Absolute Error: {mae:.2f}\n")
        f.write(f"Root Mean Squared Error: {rmse:.2f}\n")
        f.write("\nModel Architecture:\n")
        model.summary(print_fn=lambda x: f.write(x + '\n'))
    
    # Save the model
    model.save('tomato_counting_model.h5')
    
    print("Training complete! Results have been saved to 'model_results.txt' and 'training_results.png'")
    print(f"Final Mean absolute error is : {mae:.2f}")
    print(f"Final root mean square error: {rmse:.2f}")

if __name__ == "__main__":
    main()